# DSCI 551 Final Project Demo
## SQL + NoSQL Integrated Application (Spotify)

This notebook is the **main demo notebook** for our final project. It demonstrates:

- **SQL module**: custom CSV parser and operations on `SpotifyFeatures.csv`
- **NoSQL module**: custom JSON parser and operations on `spotify_reviews.json`
- **Scaling demonstration** on the SQL side (limited memory / chunk processing)
- **Final application** based on joining SQL + NoSQL data.

## 1. Imports & Setup

In [1]:
import math
import matplotlib.pyplot as plt

# SQL-side modules (from teammates)
import io_module
import processing
import core
from core import DataFrame

# NoSQL-side module (Eric)
from nosql_module import JSONParser, Collection   # 如果你的檔名或類別不同，調整這行

print("Modules imported.")

ImportError: cannot import name 'JSONParser' from 'nosql_module' (/Users/angus3938/Documents/USCcourses/dsci551/Project/Data2App/parser/nosql_module.py)

## 2. SQL Data Loading & Basic Operations

In this section we:

- Use our **custom CSV parser** to load `SpotifyFeatures.csv` into a `DataFrame`
- Convert it into a list of Python dictionaries for easier integration
- Demonstrate basic filtering on the SQL side.

In [ ]:
# Load full SpotifyFeatures.csv into a DataFrame
df_sql = DataFrame.from_csv("SpotifyFeatures.csv")
print("SQL DataFrame (head):")
print(df_sql.head(5))
print("Shape:", df_sql.shape())

# Convert DataFrame -> list[dict] for downstream join / application
sql_rows = []
for i in range(df_sql.row_count):
    row = {col: df_sql.data[col][i] for col in df_sql.columns}
    sql_rows.append(row)

print(f"Total SQL rows: {len(sql_rows)}")
sql_rows[:5]

In [ ]:
# Example: basic SQL-side filtering
# Keep tracks with popularity > 80

popular_tracks = [row for row in sql_rows if float(row.get("popularity", 0)) > 80]
print(f"Popular tracks (>80 popularity): {len(popular_tracks)}")
popular_tracks[:5]

## 3. Scaling Demo (SQL Side)

To demonstrate handling **large datasets under limited memory**, we implement
a chunk-based CSV reader on the SQL side. The idea:

- Assume the full dataset would not fit in memory at once.
- Process the file in small chunks (e.g., 500 rows at a time).
- Perform aggregation incrementally.

Here we use our `read_csv_chunks` helper, which yields `DataFrame` chunks.

In [ ]:
from io_module import read_csv_chunks

def compute_global_avg_popularity(filepath, chunk_size=1000):
    """
    Compute global average popularity using chunk-based reading,
    simulating limited memory processing.
    """
    total_pop = 0.0
    count = 0
    
    for chunk_df in read_csv_chunks(filepath, chunk_size=chunk_size):
        # chunk_df is a core.DataFrame
        pops = chunk_df.data.get("popularity", [])
        for p in pops:
            total_pop += float(p)
            count += 1
    
    return total_pop / count if count else math.nan

avg_popularity_chunks = compute_global_avg_popularity("SpotifyFeatures.csv", chunk_size=500)
print("Global average popularity (via chunks):", avg_popularity_chunks)

## 4. NoSQL Data Loading & Basic Operations

Here we:

- Use our **custom JSON parser / Collection** to load `spotify_reviews.json`
- Perform filtering, projection, and group-by aggregation on review data.

In [ ]:
# Load spotify_reviews.json via Eric's NoSQL module
reviews_collection = Collection.from_json("spotify_reviews.json")
print(reviews_collection)
print("Total NoSQL documents:", len(reviews_collection))

In [ ]:
# Example NoSQL filtering: reviews with rating >= 4
high_rating_reviews = reviews_collection.find({"rating": {"$gte": 4}})
print("High rating reviews:", high_rating_reviews.count())
high_rating_reviews.limit(5).to_list()

In [ ]:
# Example NoSQL projection: keep only track_name, rating, play_count
proj_reviews = high_rating_reviews.project(["track_name", "rating", "play_count"])
proj_reviews.limit(5).to_list()

In [ ]:
# Example NoSQL group & aggregate: average rating per track
grouped_ratings = reviews_collection.group(
    group_by="track_name",
    aggregations={
        "avg_rating": {"$avg": "rating"},
        "review_count": {"$count": "rating"}
    }
)
grouped_ratings.to_list()[:10]

## 5. Joining SQL and NoSQL Data

We now **join** the structured Spotify feature data (SQL) with our semi-structured
review data (NoSQL) using `track_name` as the join key.

In [ ]:
def join_by_track_name(sql_rows, reviews_collection, key="track_name"):
    """Simple inner join between SQL rows and NoSQL reviews on track_name.

    Returns a list of merged dictionaries.
    """
    # Build index from track_name -> list of review docs
    review_docs = reviews_collection.to_list()
    index = {}
    for doc in review_docs:
        k = doc.get(key)
        if k not in index:
            index[k] = []
        index[k].append(doc)

    joined = []
    for row in sql_rows:
        k = row.get(key)
        if k in index:
            for doc in index[k]:
                merged = {**row, **doc}
                joined.append(merged)
    return joined

joined_data = join_by_track_name(sql_rows, reviews_collection)
print("Joined records:", len(joined_data))
joined_data[:5]

## 6. Application: Spotify Track Insights

Using the joined dataset, we explore:

1. **How does user rating correlate with track popularity?**
2. **Which tracks are underrated (high rating but low popularity)?**
3. **How does audio energy relate to user satisfaction (rating)?**

In [ ]:
from collections import defaultdict

rating_sum = defaultdict(float)
rating_count = defaultdict(int)
popularity_by_track = {}
energy_by_track = {}

for row in joined_data:
    name = row.get("track_name")
    rating = float(row.get("rating", 0))
    pop = float(row.get("popularity", 0))
    energy = float(row.get("energy", 0))

    rating_sum[name] += rating
    rating_count[name] += 1
    popularity_by_track[name] = pop
    energy_by_track[name] = energy

summary = []
for name in rating_sum:
    avg_rating = rating_sum[name] / rating_count[name]
    summary.append({
        "track_name": name,
        "avg_rating": avg_rating,
        "popularity": popularity_by_track.get(name, 0),
        "energy": energy_by_track.get(name, 0),
        "review_count": rating_count[name]
    })

summary[:10]

In [ ]:
# Plot: popularity vs average rating
names = [row["track_name"] for row in summary]
avg_ratings = [row["avg_rating"] for row in summary]
popularities = [row["popularity"] for row in summary]

plt.figure(figsize=(8, 5))
plt.scatter(popularities, avg_ratings)
plt.xlabel("Popularity")
plt.ylabel("Average Rating")
plt.title("Popularity vs Average User Rating")
plt.grid(True)
plt.show()

In [ ]:
# Identify potential 'underrated' tracks: high rating but lower popularity
underrated = [
    row for row in summary
    if row["avg_rating"] >= 4.5 and row["popularity"] < 70
]
sorted(underrated, key=lambda x: (-x["avg_rating"], x["popularity"]))[:10]

## 7. Conclusion & Future Work

- We implemented custom SQL and NoSQL parsers to process Spotify feature data and user reviews.
- Using chunk-based processing, our system can scale to larger CSV files under limited memory.
- By joining structured audio features with semi-structured reviews, we built a small analytics
  application that explores relationships between popularity, energy, and user ratings.

_This notebook is intended as the main demo artifact for the final project presentation._